In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, coalesce, lit, expr, split, lpad, concat,
    sum as spark_sum, avg, abs as spark_abs, round as spark_round
)
from pyspark.sql.window import Window

# =====================================================
# LOAD TABLES
# =====================================================
ili = spark.table("bronze_catalog.bronze_stage_schema.invoice_line_items")
inv = spark.table("bronze_catalog.bronze_stage_schema.invoices")
pay = spark.table("bronze_catalog.bronze_stage_schema.payments")
prod = spark.table("bronze_catalog.bronze_stage_schema.products")
fx = spark.table("bronze_catalog.bronze_raw_schema.exchange_rates")

print("✅ Tables loaded successfully")
print(f"Invoice Line Items: {ili.count():,} rows")
print(f"Invoices: {inv.count():,} rows")
print(f"Payments: {pay.count():,} rows")
print(f"Products: {prod.count():,} rows")
print(f"Exchange Rates: {fx.count():,} rows")

# =====================================================
# PREPARE DATE FORMATS
# =====================================================
# Parse invoice_date from yyyy/M/d format
inv = inv.withColumn("inv_date_parts", split(col("invoice_date"), "/"))
inv = inv.withColumn(
    "invoice_date_normalized",
    concat(
        col("inv_date_parts")[0],
        lit("-"),
        lpad(col("inv_date_parts")[1], 2, "0"),
        lit("-"),
        lpad(col("inv_date_parts")[2], 2, "0")
    )
).withColumn(
    "invoice_date_clean",
    expr("to_date(invoice_date_normalized, 'yyyy-MM-dd')")
)

# Parse exchange rate date
fx = fx.withColumn("fx_date_parts", split(col("date"), "/"))
fx = fx.withColumn(
    "date_normalized",
    concat(
        col("fx_date_parts")[0],
        lit("-"),
        lpad(col("fx_date_parts")[1], 2, "0"),
        lit("-"),
        lpad(col("fx_date_parts")[2], 2, "0")
    )
).withColumn(
    "date_clean",
    expr("to_date(date_normalized, 'yyyy-MM-dd')")
)

# =====================================================
# JOIN TABLES
# =====================================================
# Start with invoice line items
df = ili

# Join invoices (for date, currency, status)
df = df.join(
    inv.select(
        "invoice_id",
        "invoice_date_clean",
        col("currency").alias("inv_currency"),
        "invoice_status"
    ),
    "invoice_id",
    "left"
)

# Join products (for cost_price)
df = df.join(
    prod.select("product_id", "cost_price"),
    "product_id",
    "left"
)

# Join exchange rates (for currency conversion)
df = df.join(
    fx.select("date_clean", col("currency").alias("fx_currency"), "rate_to_usd"),
    (col("invoice_date_clean") == col("date_clean")) & (col("inv_currency") == col("fx_currency")),
    "left"
)

# Join payments (for payment reconciliation)
df = df.join(
    pay.select("invoice_id", "payment_amount"),
    "invoice_id",
    "left"
)

print("\n✅ All joins completed")
print(f"Total rows after joins: {df.count():,}")
display(df.select("invoice_line_id", "invoice_id", "product_id", "quantity", "unit_price", "discount", "invoice_status", "inv_currency", "rate_to_usd").limit(10))

In [0]:
# =====================================================
# 1️⃣ HANDLE NEGATIVE QUANTITY
# =====================================================
# Logic:
# - If quantity is NULL -> set to 0
# - If quantity < 0 and invoice_status = "cancel" -> set to 0
# - If quantity < 0 and invoice_status != "cancel" -> take absolute value
# - Otherwise -> keep original quantity

df = df.withColumn(
    "quantity_fixed",
    when(col("quantity").isNull(), 0)
    .when(
        (col("quantity") < 0) & (col("invoice_status") == "cancel"),
        0
    )
    .when(
        col("quantity") < 0,
        spark_abs(col("quantity"))
    )
    .otherwise(col("quantity"))
)

print("✅ Negative quantity handling complete")

# Show statistics
qty_stats = df.groupBy(
    when(col("quantity").isNull(), "NULL")
    .when(col("quantity") < 0, "Negative")
    .when(col("quantity") == 0, "Zero")
    .otherwise("Positive")
    .alias("quantity_type")
).count().orderBy("quantity_type")

print("\nQuantity Distribution:")
display(qty_stats)

print("\nSample of fixed quantities:")
display(
    df.select(
        "invoice_line_id",
        "invoice_id",
        "invoice_status",
        col("quantity").alias("original_qty"),
        "quantity_fixed"
    ).limit(20)
)

In [0]:
# =====================================================
# 2️⃣ FIX NULL UNIT PRICE
# =====================================================
# Rule: cost_price + 25% margin
# Fallback: median unit_price by product_id

# Step 1: Use cost_price * 1.25 where unit_price is NULL
df = df.withColumn(
    "unit_price_step1",
    when(
        col("unit_price").isNull(),
        col("cost_price") * 1.25
    ).otherwise(col("unit_price"))
)

# Step 2: Calculate median unit_price by product_id for fallback
product_median = df.filter(col("unit_price_step1").isNotNull()) \
    .groupBy("product_id") \
    .agg(expr("percentile_approx(unit_price_step1, 0.5)").alias("median_price"))

df = df.join(product_median, "product_id", "left")

# Step 3: Apply fallback if still NULL
df = df.withColumn(
    "unit_price_fixed",
    coalesce(col("unit_price_step1"), col("median_price"))
)

print("✅ NULL unit_price fixing complete")

# Statistics
price_fix_stats = df.groupBy(
    when(col("unit_price").isNull(), "Fixed (was NULL)")
    .otherwise("Original")
    .alias("price_source")
).count()

print("\nUnit Price Fix Statistics:")
display(price_fix_stats)

print("\nSample of fixed prices:")
display(
    df.select(
        "invoice_line_id",
        "product_id",
        col("unit_price").alias("original_price"),
        "cost_price",
        "unit_price_fixed"
    ).limit(20)
)

In [0]:
# =====================================================
# 3️⃣ HANDLE NULL DISCOUNT USING PAYMENT
# =====================================================

# Step 1: Fill NULL discounts with 0 temporarily
df = df.withColumn(
    "temp_discount",
    coalesce(col("discount"), lit(0.0))
)

# Step 2: Calculate gross amount (before discount)
df = df.withColumn(
    "gross_local",
    col("quantity_fixed") * col("unit_price_fixed")
)

# Step 3: Calculate line amount with temp discount
df = df.withColumn(
    "line_local",
    col("gross_local") * (1 - col("temp_discount"))
)

# Step 4: Convert to USD (handles FX fluctuation)
df = df.withColumn(
    "line_usd",
    col("line_local") * coalesce(col("rate_to_usd"), lit(1.0))
)

# Step 5: Calculate invoice total in USD
from pyspark.sql.window import Window

invoice_window = Window.partitionBy("invoice_id")

df = df.withColumn(
    "invoice_total_usd",
    spark_sum("line_usd").over(invoice_window)
)

# Step 6: Convert payment to USD
df = df.withColumn(
    "payment_usd",
    col("payment_amount") * coalesce(col("rate_to_usd"), lit(1.0))
)

# Step 7: Back-calculate discount from payment for NULL discounts
# Formula: discount = 1 - (payment_usd / invoice_total_usd)
df = df.withColumn(
    "discount_calculated",
    when(
        col("discount").isNull() & col("payment_usd").isNotNull() & (col("invoice_total_usd") > 0),
        1 - (col("payment_usd") / col("invoice_total_usd"))
    ).otherwise(col("temp_discount"))
)

# Step 8: Apply safety bounds (0 to 0.9) and round to 2 decimals
df = df.withColumn(
    "discount_fixed",
    spark_round(
        when(col("discount_calculated") < 0, 0.0)
        .when(col("discount_calculated") > 0.9, 0.9)
        .otherwise(col("discount_calculated")),
        2
    )
)

print("✅ NULL discount handling complete")

# Statistics
discount_stats = df.groupBy(
    when(col("discount").isNull(), "Calculated from Payment")
    .otherwise("Original")
    .alias("discount_source")
).count()

print("\nDiscount Source Statistics:")
display(discount_stats)

print("\nSample of discount calculations:")
display(
    df.select(
        "invoice_line_id",
        "invoice_id",
        col("discount").alias("original_discount"),
        "gross_local",
        "payment_usd",
        "invoice_total_usd",
        "discount_fixed"
    ).limit(20)
)

In [0]:
# =====================================================
# VALIDATION: ANALYZE NULL DISCOUNT ISSUE (25.2%)
# =====================================================

print("🔍 INVESTIGATING NULL DISCOUNT ROOT CAUSES\n")
print("=" * 80)

# 1. Check payment coverage
print("\n1️⃣ PAYMENT COVERAGE ANALYSIS")
print("-" * 80)

payment_coverage = df.groupBy(
    when(col("discount").isNull(), "NULL Discount")
    .otherwise("Has Discount")
    .alias("discount_status"),
    when(col("payment_amount").isNull(), "No Payment")
    .otherwise("Has Payment")
    .alias("payment_status")
).count().orderBy("discount_status", "payment_status")

print("\nDiscount vs Payment Coverage:")
display(payment_coverage)

# 2. Analyze discount distribution
print("\n2️⃣ CALCULATED DISCOUNT DISTRIBUTION")
print("-" * 80)

calc_discount_dist = df.filter(col("discount").isNull()).groupBy(
    when(col("discount_fixed") == 0, "0% (No discount)")
    .when(col("discount_fixed") < 0.1, "1-9%")
    .when(col("discount_fixed") < 0.3, "10-29%")
    .when(col("discount_fixed") < 0.5, "30-49%")
    .when(col("discount_fixed") < 0.7, "50-69%")
    .otherwise("70-90% (HIGH!)")
    .alias("discount_range")
).count().orderBy("discount_range")

print("\nCalculated Discount Distribution:")
display(calc_discount_dist)

# 3. Identify high-discount invoices (potential issues)
print("\n3️⃣ HIGH CALCULATED DISCOUNTS (>50%)")
print("-" * 80)

high_discount_invoices = df.filter(
    col("discount").isNull() & (col("discount_fixed") > 0.5)
).select(
    "invoice_id",
    "invoice_status",
    "inv_currency",
    "gross_local",
    "payment_usd",
    "invoice_total_usd",
    col("discount_fixed").alias("calculated_discount")
).distinct().orderBy(col("calculated_discount").desc())

high_discount_count = high_discount_invoices.count()
print(f"\n⚠️  Found {high_discount_count} invoices with calculated discount >50%")
if high_discount_count > 0:
    print("\nTop 10 High Discount Invoices:")
    display(high_discount_invoices.limit(10))
else:
    print("✅ No concerning high discounts found.")

# 4. Analyze by invoice status
print("\n4️⃣ NULL DISCOUNT BY INVOICE STATUS")
print("-" * 80)

status_analysis = df.filter(col("discount").isNull()).groupBy("invoice_status").agg(
    expr("count(*)").alias("null_discount_count"),
    expr("avg(discount_fixed)").alias("avg_calculated_discount"),
    expr("count(distinct invoice_id)").alias("unique_invoices")
).orderBy(col("null_discount_count").desc())

print("\nNULL Discount by Invoice Status:")
display(status_analysis)

print("\n" + "=" * 80)
print("\n💡 RECOMMENDATIONS:")
print("   1. Check if 'NULL discount' means 'no discount' (business rule)")
print("   2. Verify high-discount invoices aren't partial payments")
print("   3. Consider adding a flag: 'discount_source' (original vs calculated)")
print("   4. Investigate invoices with missing payment data")
print("=" * 80)

In [0]:
# =====================================================
# SOLUTION: ADD QUALITY FLAGS & IMPROVED LOGIC
# =====================================================

print("🔧 APPLYING IMPROVEMENTS TO DISCOUNT LOGIC\n")
print("=" * 80)

# 1. Add discount_source flag
df = df.withColumn(
    "discount_source",
    when(col("discount").isNotNull(), "original")
    .when(col("payment_usd").isNotNull(), "calculated_from_payment")
    .otherwise("default_zero")
)

print("✅ Step 1: Added 'discount_source' flag")

# 2. Add data quality flag for suspicious records
df = df.withColumn(
    "data_quality_flag",
    when(
        col("discount").isNull() & col("payment_usd").isNull(),
        "WARN: Missing discount and payment"
    )
    .when(
        col("discount").isNull() & (col("discount_fixed") > 0.7),
        "WARN: High calculated discount (>70%)"
    )
    .when(
        col("discount").isNull() & (col("discount_fixed") > 0.5),
        "CAUTION: Calculated discount >50%"
    )
    .when(
        (col("quantity") < 0) | (col("unit_price").isNull()),
        "INFO: Fixed negative qty or NULL price"
    )
    .otherwise("OK")
)

print("✅ Step 2: Added 'data_quality_flag' for suspicious records")

# 3. Add confidence score for calculated discounts
df = df.withColumn(
    "discount_confidence",
    when(
        col("discount").isNotNull(),
        lit(1.0)  # Original discount = 100% confidence
    )
    .when(
        col("discount").isNull() & col("payment_usd").isNotNull() & (col("discount_fixed") <= 0.5),
        lit(0.8)  # Calculated, reasonable discount = 80% confidence
    )
    .when(
        col("discount").isNull() & col("payment_usd").isNotNull() & (col("discount_fixed") > 0.5),
        lit(0.5)  # Calculated, high discount = 50% confidence
    )
    .otherwise(lit(0.3))  # Default zero or missing payment = 30% confidence
)

print("✅ Step 3: Added 'discount_confidence' score (0.0-1.0)")

# 4. Alternative: Assume NULL discount = 0% (no discount) instead of payment reconciliation
df = df.withColumn(
    "discount_alternative",
    when(
        col("discount").isNotNull(),
        col("discount")
    )
    .otherwise(lit(0.0))  # Treat NULL as zero discount
)

print("✅ Step 4: Added 'discount_alternative' (treats NULL as 0% discount)")

print("\n" + "=" * 80)

# Summary of quality flags
quality_summary = df.groupBy("data_quality_flag").count().orderBy(col("count").desc())

print("\n📊 DATA QUALITY SUMMARY:")
display(quality_summary)

# Discount source breakdown
discount_source_summary = df.groupBy("discount_source").agg(
    expr("count(*)").alias("record_count"),
    expr("avg(discount_fixed)").alias("avg_discount"),
    expr("avg(discount_confidence)").alias("avg_confidence")
).orderBy(col("record_count").desc())

print("\n📊 DISCOUNT SOURCE BREAKDOWN:")
display(discount_source_summary)

print("\n" + "=" * 80)
print("\n🎯 NEXT STEPS:")
print("   • Review records with 'WARN' or 'CAUTION' flags")
print("   • Consult business team: Does NULL discount mean 0% or missing data?")
print("   • Consider using 'discount_alternative' if NULL = no discount")
print("   • Use 'discount_confidence' to filter low-quality calculations")
print("=" * 80)

In [0]:
# =====================================================
# 4️⃣ CALCULATE FINAL AMOUNTS
# =====================================================

# Final amount in local currency (rounded to 2 decimals)
df = df.withColumn(
    "final_local_amount",
    spark_round(
        col("quantity_fixed") * col("unit_price_fixed") * (1 - col("discount_fixed")),
        2
    )
)

# Final amount in USD (rounded to 2 decimals)
df = df.withColumn(
    "final_usd_amount",
    spark_round(
        col("final_local_amount") * coalesce(col("rate_to_usd"), lit(1.0)),
        2
    )
)

print("✅ Final amount calculations complete (rounded to 2 decimals)")

# Summary statistics
summary = df.select(
    spark_sum("final_local_amount").alias("total_local"),
    spark_sum("final_usd_amount").alias("total_usd"),
    avg("discount_fixed").alias("avg_discount"),
    expr("count(distinct invoice_id)").alias("unique_invoices"),
    expr("count(distinct product_id)").alias("unique_products")
).collect()[0]

print(f"\n┌── SUMMARY STATISTICS ─────────────────────────────────────────────────────────")
print(f"│ Total Amount (Local Currency): ${summary['total_local']:,.2f}")
print(f"│ Total Amount (USD): ${summary['total_usd']:,.2f}")
print(f"│ Average Discount: {summary['avg_discount']:.4f} ({summary['avg_discount']*100:.2f}%)")
print(f"│ Unique Invoices: {summary['unique_invoices']:,}")
print(f"│ Unique Products: {summary['unique_products']:,}")
print("└───────────────────────────────────────────────────────────────────────────────")

print("\nSample of final amounts:")
display(
    df.select(
        "invoice_line_id",
        "invoice_id",
        "product_id",
        "quantity_fixed",
        "unit_price_fixed",
        "discount_fixed",
        "inv_currency",
        "rate_to_usd",
        "final_local_amount",
        "final_usd_amount"
    ).limit(20)
)

In [0]:
# =====================================================
# FINAL CLEAN TABLE WITH QUALITY INDICATORS
# =====================================================
# Enhanced schema with quality tracking:
# - Original Task-1 columns (invoice_line_id, invoice_id, product_id, etc.)
# - discount_source: tracks origin of discount value
# - discount_confidence: 0.0-1.0 confidence score
# - data_quality_flag: identifies suspicious records
# - discount_alternative: alternative calculation (NULL = 0%)

invoice_line_items_clean = df.select(
    "invoice_line_id",
    "invoice_id",
    "product_id",
    col("quantity_fixed").alias("quantity"),
    col("unit_price_fixed").alias("unit_price"),
    col("discount_fixed").alias("discount"),
    col("inv_currency").alias("currency"),
    "rate_to_usd",
    "final_local_amount",
    "final_usd_amount",
    # Quality tracking columns
    "discount_source",
    "discount_confidence",
    "discount_alternative",
    "data_quality_flag"
)

print("✅ Enhanced clean silver layer table created successfully!")
print(f"Total Records: {invoice_line_items_clean.count():,}")

print("\n┌── EDGE CASES HANDLED ────────────────────────────────────────────────────────────")
print("│ ✔ Negative quantity: status-aware correction")
print("│ ✔ NULL unit_price: cost_price + 25% margin + median fallback")
print("│ ✔ NULL discount: payment reconciliation + quality flags")
print("│ ✔ FX fluctuation: invoice_date join")
print("│ ✔ Invalid discounts: clipped to [0, 0.9]")
print("│ ✔ Rounding: discount rounded to 2 decimals")
print("│ ✔ Missing quantity: set to 0")
print("│ ✔ Final amounts: rounded to 2 decimals")
print("└────────────────────────────────────────────────────────────────────────────────")

print("\n┌── NEW QUALITY COLUMNS ────────────────────────────────────────────────────────────")
print("│ 📊 discount_source: Tracks if discount is original/calculated/default")
print("│ 📊 discount_confidence: 0.0-1.0 confidence score for discount value")
print("│ 📊 discount_alternative: Alternative logic (NULL = 0% discount)")
print("│ 📊 data_quality_flag: Flags suspicious records (WARN/CAUTION/OK)")
print("└────────────────────────────────────────────────────────────────────────────────")

print("\n📊 Enhanced Clean Invoice Line Items (with Quality Tracking):")
display(
    invoice_line_items_clean
    .orderBy("invoice_id", "invoice_line_id")
    .limit(30)
)

# Show quality flag distribution
print("\n⚠️  DATA QUALITY FLAG DISTRIBUTION:")
quality_dist = invoice_line_items_clean.groupBy("data_quality_flag").count().orderBy(col("count").desc())
display(quality_dist)

# Display table schema
print("\n" + "="*80)
print("\n📝 FINAL TABLE SCHEMA:")
print("\n" + "="*80)
invoice_line_items_clean.printSchema()
print("="*80)

print("\n📋 SCHEMA SUMMARY:")
print("-" * 80)
for field in invoice_line_items_clean.schema.fields:
    nullable = "nullable" if field.nullable else "not null"
    print(f"  {field.name:30s} | {str(field.dataType):20s} | {nullable}")
print("="*80)

# Uncomment to save to silver layer with quality columns
# invoice_line_items_clean.write.mode("overwrite").saveAsTable(
#     "silver_catalog.silver_schema.invoice_line_items_enhanced"
# )
# print("\n✅ Enhanced table saved to: silver_catalog.silver_schema.invoice_line_items_enhanced")

In [0]:
# =====================================================
# SAVE TO SILVER CATALOG
# =====================================================

print("💾 Saving enhanced invoice_line_items table to Silver Catalog...\n")

# Define target table path
target_table = "silver_catalog.silver_schema.invoice_line_items"

try:
    # Write to silver catalog
    invoice_line_items_clean.write.mode("overwrite").saveAsTable(target_table)
    
    print("✅ SUCCESS! Table saved successfully.")
    print("=" * 80)
    print(f"📊 Table: {target_table}")
    print(f"📊 Total Records: {invoice_line_items_clean.count():,}")
    print(f"📊 Columns: {len(invoice_line_items_clean.columns)}")
    print("=" * 80)
    
    # Verify the save
    print("\n🔍 Verifying saved table...")
    saved_table = spark.table(target_table)
    print(f"✅ Verified: {saved_table.count():,} records in silver catalog")
    
    print("\n📄 Table Columns:")
    for col_name in saved_table.columns:
        print(f"   • {col_name}")
    
    print("\n✨ Table ready for downstream consumption!")
    
except Exception as e:
    print(f"❌ ERROR: Failed to save table")
    print(f"Error details: {str(e)}")
    raise

In [0]:
# =====================================================
# SAVE TO SILVER CATALOG
# =====================================================

print("📦 Saving enhanced table to Silver Catalog...\n")
print("="*80)

target_table = "silver_catalog.silver_schema.invoice_line_items"

# Write to silver layer with overwrite mode
invoice_line_items_clean.write.mode("overwrite").saveAsTable(target_table)

print(f"✅ Table successfully saved to: {target_table}")
print(f"\n📊 Total Records Written: {invoice_line_items_clean.count():,}")
print(f"📋 Total Columns: {len(invoice_line_items_clean.columns)}")

print("\n" + "="*80)
print("\n📄 TABLE DETAILS:")
print("-" * 80)
print(f"  Catalog:  silver_catalog")
print(f"  Schema:   silver_schema")
print(f"  Table:    invoice_line_items_enhanced")
print("-" * 80)

print("\n✅ COLUMNS SAVED:")
for idx, col_name in enumerate(invoice_line_items_clean.columns, 1):
    print(f"  {idx:2d}. {col_name}")

print("\n" + "="*80)
print("\n✨ Silver layer table is now ready for downstream consumption!")
print("="*80)

# Verify the table was created
print("\n🔍 Verifying table creation...")
verify_df = spark.table(target_table)
print(f"✅ Verification successful! Table exists with {verify_df.count():,} records.")

print("\n📝 Quick preview of saved table:")
display(verify_df.limit(10))